# DCGAN on MNIST (PyTorch)

A Deep Convolutional GAN trained on MNIST. Includes:
- Generator & Discriminator (DCGAN architecture)
- Training loop with the standard non-saturating GAN loss
- `wandb` logging (losses, generated-image grids)
- `tqdm` progress bars

Reference: Radford et al., *Unsupervised Representation Learning with Deep Convolutional GANs* (2015).

## 1. Setup & imports

In [ ]:
# !pip install torch torchvision wandb tqdm matplotlib

In [2]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from torchvision.utils import make_grid
from tqdm.auto import tqdm
import wandb

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


## 2. Hyperparameters

In [3]:
config = dict(
    image_size=64,      # DCGAN upsamples MNIST 28x28 -> 64x64
    channels=1,         # MNIST is grayscale
    z_dim=100,          # latent vector size
    g_features=64,      # base feature-map count for generator
    d_features=64,      # base feature-map count for discriminator
    batch_size=128,
    epochs=30,
    lr=2e-4,
    beta1=0.5,          # Adam betas from the DCGAN paper
    beta2=0.999,
    seed=42,
)

torch.manual_seed(config["seed"])

## 3. Data

Resize to 64x64 and normalize to `[-1, 1]` so the inputs match the generator's `tanh` output range.

In [4]:
transform = transforms.Compose([
    transforms.Resize(config["image_size"]),
    transforms.ToTensor(),
    transforms.Normalize([0.5], [0.5]),  # [0,1] -> [-1,1]
])

dataset = datasets.MNIST(root="./data", train=True, download=True, transform=transform)
dataloader = DataLoader(
    dataset,
    batch_size=config["batch_size"],
    shuffle=True,
    num_workers=2,
    pin_memory=True,
    drop_last=True,
)
print(f"Dataset size: {len(dataset)} | batches/epoch: {len(dataloader)}")

Dataset size: 60000 | batches/epoch: 468


## 4. Models

**Generator**: project the latent vector and upsample through `ConvTranspose2d` layers `1 -> 4 -> 8 -> 16 -> 32 -> 64`.

**Discriminator**: mirror of the generator with strided `Conv2d` layers downsampling `64 -> 32 -> 16 -> 8 -> 4 -> 1`.

In [5]:
class Generator(nn.Module):
    def __init__(self, z_dim, channels, g):
        super().__init__()
        self.net = nn.Sequential(
            # z_dim x 1 x 1 -> (g*8) x 4 x 4
            self._block(z_dim, g * 8, 4, 1, 0),
            # -> (g*4) x 8 x 8
            self._block(g * 8, g * 4, 4, 2, 1),
            # -> (g*2) x 16 x 16
            self._block(g * 4, g * 2, 4, 2, 1),
            # -> g x 32 x 32
            self._block(g * 2, g, 4, 2, 1),
            # -> channels x 64 x 64
            nn.ConvTranspose2d(g, channels, 4, 2, 1),
            nn.Tanh(),
        )

    @staticmethod
    def _block(in_c, out_c, k, s, p):
        return nn.Sequential(
            nn.ConvTranspose2d(in_c, out_c, k, s, p, bias=False),
            nn.BatchNorm2d(out_c),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.net(x)


class Discriminator(nn.Module):
    def __init__(self, channels, d):
        super().__init__()
        self.net = nn.Sequential(
            # channels x 64 x 64 -> d x 32 x 32
            nn.Conv2d(channels, d, 4, 2, 1),
            nn.LeakyReLU(0.2, inplace=True),
            # -> (d*2) x 16 x 16
            self._block(d, d * 2, 4, 2, 1),
            # -> (d*4) x 8 x 8
            self._block(d * 2, d * 4, 4, 2, 1),
            # -> (d*8) x 4 x 4
            self._block(d * 4, d * 8, 4, 2, 1),
            # -> 1 x 1 x 1
            nn.Conv2d(d * 8, 1, 4, 1, 0),
        )

    @staticmethod
    def _block(in_c, out_c, k, s, p):
        return nn.Sequential(
            nn.Conv2d(in_c, out_c, k, s, p, bias=False),
            nn.BatchNorm2d(out_c),
            nn.LeakyReLU(0.2, inplace=True),
        )

    def forward(self, x):
        return self.net(x).view(-1)  # logits, shape (N,)


def weights_init(m):
    """DCGAN weight init: N(0, 0.02)."""
    classname = m.__class__.__name__
    if "Conv" in classname:
        nn.init.normal_(m.weight.data, 0.0, 0.02)
    elif "BatchNorm" in classname:
        nn.init.normal_(m.weight.data, 1.0, 0.02)
        nn.init.constant_(m.bias.data, 0)

In [6]:
gen = Generator(config["z_dim"], config["channels"], config["g_features"]).to(device)
disc = Discriminator(config["channels"], config["d_features"]).to(device)
gen.apply(weights_init)
disc.apply(weights_init)

n_params = lambda m: sum(p.numel() for p in m.parameters())
print(f"Generator params:     {n_params(gen):,}")
print(f"Discriminator params: {n_params(disc):,}")

Generator params:     3,574,657
Discriminator params: 2,763,585


## 5. Loss & optimizers

`BCEWithLogitsLoss` (numerically stable) with the standard non-saturating generator objective. Separate Adam optimizers for G and D.

In [7]:
criterion = nn.BCEWithLogitsLoss()
opt_gen = optim.Adam(gen.parameters(), lr=config["lr"], betas=(config["beta1"], config["beta2"]))
opt_disc = optim.Adam(disc.parameters(), lr=config["lr"], betas=(config["beta1"], config["beta2"]))

# Fixed noise to visualize generator progress across epochs.
fixed_noise = torch.randn(64, config["z_dim"], 1, 1, device=device)

## 6. wandb init

In [8]:
# Run `wandb login` first, or set mode="offline"/"disabled" below to skip cloud logging.
wandb.init(project="dcgan-mnist", config=config)
wandb.watch(gen, log="all", log_freq=200)

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from C:\Users\Tonkid\_netrc.
wandb: Currently logged in as: rtw-cckp (rtw-cckp-cmkl-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


## 7. Training loop

In [ ]:
step = 0
for epoch in range(config["epochs"]):
    pbar = tqdm(dataloader, desc=f"Epoch {epoch + 1}/{config['epochs']}", leave=True)
    for real, _ in pbar:
        real = real.to(device)
        bs = real.size(0)
        noise = torch.randn(bs, config["z_dim"], 1, 1, device=device)
        fake = gen(noise)

        # ---- Train Discriminator: max log(D(x)) + log(1 - D(G(z))) ----
        disc_real = disc(real)
        loss_disc_real = criterion(disc_real, torch.ones_like(disc_real))
        disc_fake = disc(fake.detach())
        loss_disc_fake = criterion(disc_fake, torch.zeros_like(disc_fake))
        loss_disc = (loss_disc_real + loss_disc_fake) / 2

        opt_disc.zero_grad()
        loss_disc.backward()
        opt_disc.step()

        # ---- Train Generator: max log(D(G(z))) (non-saturating) ----
        output = disc(fake)
        loss_gen = criterion(output, torch.ones_like(output))

        opt_gen.zero_grad()
        loss_gen.backward()
        opt_gen.step()

        # ---- Logging ----
        if step % 50 == 0:
            with torch.no_grad():
                D_x = torch.sigmoid(disc_real).mean().item()
                D_G_z = torch.sigmoid(disc_fake).mean().item()
            pbar.set_postfix(loss_d=f"{loss_disc.item():.3f}",
                             loss_g=f"{loss_gen.item():.3f}",
                             D_x=f"{D_x:.2f}", D_Gz=f"{D_G_z:.2f}")
            wandb.log({
                "loss/discriminator": loss_disc.item(),
                "loss/generator": loss_gen.item(),
                "D(x)": D_x,
                "D(G(z))": D_G_z,
                "epoch": epoch,
            }, step=step)
        step += 1

    # ---- End of epoch: log a grid of generated samples ----
    gen.eval()
    with torch.no_grad():
        samples = gen(fixed_noise)
    gen.train()
    grid = make_grid(samples, nrow=8, normalize=True, value_range=(-1, 1))
    wandb.log({"generated": wandb.Image(grid, caption=f"epoch {epoch + 1}")}, step=step)

print("Training complete.")

Epoch 1/30:   0%|          | 0/468 [00:00<?, ?it/s]

Epoch 2/30:   0%|          | 0/468 [00:00<?, ?it/s]

Epoch 3/30:   0%|          | 0/468 [00:00<?, ?it/s]

Epoch 4/30:   0%|          | 0/468 [00:00<?, ?it/s]

Epoch 5/30:   0%|          | 0/468 [00:00<?, ?it/s]

...

Epoch 9/30:   0%|          | 0/468 [00:00<?, ?it/s]

Epoch 10/30:   0%|          | 0/468 [00:00<?, ?it/s]

Epoch 11/30:   0%|          | 0/468 [00:00<?, ?it/s]

Epoch 12/30:   0%|          | 0/468 [00:00<?, ?it/s]

Epoch 13/30:   0%|          | 0/468 [00:00<?, ?it/s]

## 8. Save & finish

In [ ]:
os.makedirs("checkpoints", exist_ok=True)
torch.save(gen.state_dict(), "checkpoints/generator.pth")
torch.save(disc.state_dict(), "checkpoints/discriminator.pth")
wandb.finish()
print("Saved models to ./checkpoints/")

## 9. Generate samples from the trained model

In [ ]:
import matplotlib.pyplot as plt

gen.eval()
with torch.no_grad():
    noise = torch.randn(64, config["z_dim"], 1, 1, device=device)
    samples = gen(noise)
grid = make_grid(samples, nrow=8, normalize=True, value_range=(-1, 1)).cpu()

plt.figure(figsize=(8, 8))
plt.axis("off")
plt.title("DCGAN-generated MNIST digits")
plt.imshow(grid.permute(1, 2, 0).squeeze(), cmap="gray")
plt.show()